In [1]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.insert(0, '..')  # nbs/notebook_cache.py
from notebook_cache import NotebookCache

cache = NotebookCache.here()

# Verne, small, step 02: cosine learning rate (roadmap box 11), on T1 and T1-wide

What's new: `02_cosine_lr_explained.ipynb`. Step 01: `01_canvas.ipynb`.

One change, in training only (`train.py`): the learning rate decays from 3e-4 to 0 along a cosine over the 20k steps, instead of staying at 3e-4. Model, data, steps, batch: step 01's (the canvas, [top | wrist]).

Why: at a constant rate the weights keep jittering to the last step, and step 01 showed runs are fragile: T1 seeds spread 10-30 of 50, and on T1-wide only some runs find the cube (probe x slope 0.61 / ~0 / ~0 on the pod; 0.95 for the same seed on the local GPU). Decaying the rate lets a run settle. Win = smaller spread across seeds and more runs that find the cube, at an equal or better mean.

```-
lr:  step 01   3e-4 ───────────────────────────────  (constant)
     step 02   3e-4 ──╮
                       ╰──╮
                           ╰────╮
                                 ╰──────── 0        (cosine)
```

Every score: training seeds `TRAIN_SEEDS` × 50 rollout seeds (1000-1049), at n_exec 8 and 16, trained on the RunPod RTX 4090 like step 01. Step 01 (`01_canvas.ipynb`, e16af3e), seeds 0 / 1 / 2: T1 n_exec 8: 10 / 30 / 19, n_exec 16: 15 / 16 / 17; T1-wide n_exec 8: 3 / 3 / 12, n_exec 16: 3 / 10 / 8; probe x slope +0.25 / +0.03 / +0.61, miss 9.4 / 10.2 / 7.5 cm.

### step 02.1 — the data


In [2]:
import numpy as np
from frozendict import frozendict

from dream_robot.policies.verne.data import canvas, load_cameras, load_table, to_gpu

NAME = 'pick_place_cube_n100'

In [3]:
@cache
def table():
    state, action, episode = load_table(NAME)  # [N, 8], [N, 8], [N]
    return frozendict(state=state, action=action, episode=episode)

table: cached, 1 call(s) stored


In [4]:
top, wrist = load_cameras(NAME)  # [N, 3, 128, 128] uint8 each
# decoded once per dataset into data/robosuite/<NAME>/decoded/, then read in seconds
cams = frozendict(top=top, wrist=wrist)

decoding 21295 top + wrist frames once -> /home/rahul/Documents/codes/ml/dream-robot/data/robosuite/pick_place_cube_n100/decoded


In [5]:
data = table()
val = np.random.default_rng(0).permutation(100)[:10]  # as dreamzero / uwm2
rest = np.setdiff1d(np.arange(100), val)
split = frozendict(val=val, labelled=np.random.default_rng(1).choice(rest, 10, replace=False))
demos = to_gpu(
    canvas(cams['top'], cams['wrist']),
    data['state'],
    data['action'],
    data['episode'],
    split['labelled'],
)
print({k: len(v) for k, v in split.items()}, tuple(demos.canvas.shape))

{'val': 10, 'labelled': 10} (21295, 3, 128, 256)


### step 02.2 — the model


In [6]:
import torch

from dream_robot.policies.verne.model import Verne

net = Verne()
print(f'{sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params')

3.68M params


### step 02.3 — train (flow matching on the 16-action chunk, 20k steps, ~3 min each), each training seed


In [7]:
import dream_robot.policies.verne as verne


@cache
def train_run(steps: int, seed: int):
    return verne.train.train(demos, split['labelled'], split['val'], steps, seed=seed)

train_run: cached, 1 call(s) stored


In [8]:
TRAIN_SEEDS = (0,)  # results are cached per seed, so added seeds just join

In [9]:
runs = {seed: train_run(20000, seed) for seed in TRAIN_SEEDS}  # seed -> {'weights', 'log'}

step      0   train 2.224   val 2.868
step   1000   train 0.164   val 0.194
step   2000   train 0.079   val 0.167
step   3000   train 0.075   val 0.225
step   4000   train 0.043   val 0.187
step   5000   train 0.045   val 0.202
step   6000   train 0.067   val 0.247
step   7000   train 0.030   val 0.223
step   8000   train 0.031   val 0.222
step   9000   train 0.017   val 0.252
step  10000   train 0.039   val 0.284
step  11000   train 0.028   val 0.288
step  12000   train 0.027   val 0.286
step  13000   train 0.018   val 0.370
step  14000   train 0.033   val 0.352
step  15000   train 0.017   val 0.335
step  16000   train 0.011   val 0.412
step  17000   train 0.009   val 0.379
step  18000   train 0.005   val 0.367
step  19000   train 0.016   val 0.471
step  20000   train 0.002   val 0.544


### step 02.4 — rollouts: T1 seeds 1000-1049, replan after 8 or all 16 actions, each training seed


In [10]:
from dream_robot.policies.verne import rollouts


def total(scores: dict, name: str) -> None:
    # scores: (train seed, n_exec) -> rollout results; prints one line per n_exec
    for n_exec in (8, 16):
        oks = [sum(ok for ok, _, _ in scores[seed, n_exec].values()) for seed in TRAIN_SEEDS]
        n = len(rollouts.SEEDS) * len(TRAIN_SEEDS)
        print(f'{name} n_exec {n_exec:2d}: {" + ".join(map(str, oks))} = {sum(oks)}/{n}')

In [11]:
@cache(2)  # 2: one torch seed per episode, workers
def sweep(seed: int, n_exec: int) -> dict[int, tuple[bool, str, int]]:
    return rollouts.sweep(runs[seed]['weights'], demos, wide=False, n_exec=n_exec)

sweep: cached, 2 call(s) stored


In [12]:
scores = {(seed, n): sweep(seed, n) for seed in TRAIN_SEEDS for n in (8, 16)}
total(scores, 'T1')

n_exec  8:  5/50   failures {'no_grasp': 45}
n_exec 16: 14/50   failures {'no_grasp': 35, 'dropped': 1}
T1 n_exec  8: 5 = 5/50
T1 n_exec 16: 14 = 14/50


### step 02.5 — T1-wide: a cube that moves (x ±10 cm, y −10..+6 cm)

On T1 the cube spawns within ±3 cm, so a policy can grasp the average spot without looking. T1-wide (`data/robosuite/pick_place_cube_wide_n100`, expert 100/100) makes it find the cube. 90 train demos, 10 held out (the same episode ids as T1's val), 20k steps, each training seed, rollout seeds 1000-1049.


In [13]:
WIDE = 'pick_place_cube_wide_n100'


@cache
def table_wide():
    state, action, episode = load_table(WIDE)  # [N, 8], [N, 8], [N]
    return frozendict(state=state, action=action, episode=episode)

table_wide: cached, 1 call(s) stored


In [14]:
top, wrist = load_cameras(WIDE)
w_cams = frozendict(top=top, wrist=wrist)

decoding 22409 top + wrist frames once -> /home/rahul/Documents/codes/ml/dream-robot/data/robosuite/pick_place_cube_wide_n100/decoded


In [15]:
w_data = table_wide()
w_split = frozendict(val=val, train=np.setdiff1d(np.arange(100), val))
w_demos = to_gpu(
    canvas(w_cams['top'], w_cams['wrist']),
    w_data['state'],
    w_data['action'],
    w_data['episode'],
    w_split['train'],
)
print({k: len(v) for k, v in w_split.items()}, tuple(w_demos.canvas.shape))

{'val': 10, 'train': 90} (22409, 3, 128, 256)


In [16]:
@cache
def train_wide(steps: int, seed: int):
    return verne.train.train(w_demos, w_split['train'], w_split['val'], steps, seed=seed)

train_wide: cached, 1 call(s) stored


In [17]:
w_runs = {seed: train_wide(20000, seed) for seed in TRAIN_SEEDS}

step      0   train 2.231   val 2.667
step   1000   train 0.188   val 0.133
step   2000   train 0.077   val 0.100
step   3000   train 0.101   val 0.100
step   4000   train 0.046   val 0.078
step   5000   train 0.044   val 0.068
step   6000   train 0.069   val 0.075
step   7000   train 0.035   val 0.059
step   8000   train 0.043   val 0.061
step   9000   train 0.026   val 0.056
step  10000   train 0.048   val 0.064
step  11000   train 0.034   val 0.051
step  12000   train 0.030   val 0.045
step  13000   train 0.031   val 0.053
step  14000   train 0.044   val 0.041
step  15000   train 0.030   val 0.037
step  16000   train 0.026   val 0.037
step  17000   train 0.010   val 0.041
step  18000   train 0.018   val 0.035
step  19000   train 0.024   val 0.039
step  20000   train 0.015   val 0.042


In [18]:
@cache(2)  # 2: one torch seed per episode, workers
def sweep_wide(seed: int, n_exec: int) -> dict[int, tuple[bool, str, int]]:
    return rollouts.sweep(w_runs[seed]['weights'], w_demos, wide=True, n_exec=n_exec)

sweep_wide: cached, 2 call(s) stored


In [19]:
w_scores = {(seed, n): sweep_wide(seed, n) for seed in TRAIN_SEEDS for n in (8, 16)}
total(w_scores, 'T1-wide')

6 episode(s) not finished by the workers, running here: [1005, 1006, 1007, 1010, 1012, 1013]
n_exec  8: 11/50   failures {'no_grasp': 37, 'dropped': 2}
50 episode(s) not finished by the workers, running here: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049]
n_exec 16:  7/50   failures {'no_grasp': 42, 'dropped': 1}
T1-wide n_exec  8: 11 = 11/50
T1-wide n_exec 16: 7 = 7/50


### step 02.6 — the close-point probe: does the gripper close where the cube is?

For each T1-wide rollout (n_exec 8): the gripper's xy when it first commands close, against the cube's spawn xy. Slope ≈ 1 and a small miss: it goes to the cube. Slope ≈ 0: it goes to a remembered spot whatever the cube does. One line per training seed.


In [20]:
@cache(2)  # 2: one torch seed per episode, workers
def close_points_wide(seed: int, n_exec: int) -> dict:
    return rollouts.close_points(w_runs[seed]['weights'], w_demos, wide=True, n_exec=n_exec)

close_points_wide: cached, 1 call(s) stored


In [21]:
fits = {}
for seed in TRAIN_SEEDS:
    print(f'train seed {seed}: ', end='')
    fits[seed] = rollouts.close_fit(close_points_wide(seed, 8))

train seed 0: 50 episode(s) not finished by the workers, running here: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049]
closes 28/50, miss  3.0 cm, x slope +0.74 (corr +0.91, intercept -0.5 cm), y slope +0.63 (corr +0.61, intercept -1.4 cm)


### step 02.7 — as jobs on an L40S (`run.py`): T1 seed 0, T1-wide seeds 0-2

From here runs are jobs: `python -m dream_robot.policies.verne.run job 02_cosine_lr <task> <seed>` trains, sweeps and probes, and writes `experiments/verne/runs/02_cosine_lr/<task>/seed<k>/` (meta = commit + package hash + GPU). The notebook only reads those files. Seed 0 of both tasks was rerun this way on a RunPod L40S (a different GPU than the cells above, the 4090), then T1-wide seeds 1 and 2. T1 seeds 1, 2 not run.

In [22]:
import json

from dream_robot.policies.verne.run import RUNS


def read_run(step: str, task: str, seed: int) -> dict:
    # one run folder -> meta, log [(step, train, val)], n_exec -> successes, probe fit (T1-wide)
    d = RUNS / step / task / f'seed{seed}'
    sweeps = {n: json.loads((d / f'sweep_{n}.json').read_text()) for n in (8, 16)}
    probe = d / 'probe.json'
    return {
        'meta': json.loads((d / 'meta.json').read_text()),
        'log': json.loads((d / 'log.json').read_text()),
        'ok': {n: sum(r[0] for r in s.values()) for n, s in sweeps.items()},
        'fit': json.loads(probe.read_text())['fit'] if probe.exists() else None,
    }

In [23]:
for task, seeds in (('t1', (0,)), ('t1-wide', (0, 1, 2))):
    for seed in seeds:
        r = read_run('02_cosine_lr', task, seed)
        line = f'{task:8s} seed {seed} on {r["meta"]["gpu"]}: n_exec 8 {r["ok"][8]:2d}/50, '
        line += f'16 {r["ok"][16]:2d}/50, val end {r["log"][-1][2]:.3f}'
        if f := r['fit']:
            line += (
                f', probe x {f["slope_x"]:+.2f} y {f["slope_y"]:+.2f} miss {f["miss_cm"]:.1f} cm'
            )
        print(line)

t1       seed 0 on NVIDIA L40S: n_exec 8  5/50, 16 10/50, val end 0.531
t1-wide  seed 0 on NVIDIA L40S: n_exec 8  9/50, 16  4/50, val end 0.043, probe x +0.73 y +0.55 miss 3.3 cm
t1-wide  seed 1 on NVIDIA L40S: n_exec 8 11/50, 16  6/50, val end 0.031, probe x +0.05 y +0.01 miss 8.7 cm
t1-wide  seed 2 on NVIDIA L40S: n_exec 8  9/50, 16  4/50, val end 0.036, probe x +0.23 y +0.06 miss 7.6 cm


### wrap-up — cosine LR (box 11): done; T1-wide on 3 seeds, T1 on seed 0

```-
                       n_exec 8        n_exec 16       probe x slope / miss (seeds 0, 1, 2)
T1-wide  step 01 4090  3+3+12 = 18/150  3+10+8 = 21/150  +0.25 +0.03 +0.61 / 9.4 10.2 7.5 cm
         step 02 L40S  9+11+9 = 29/150  4+6+4  = 14/150  +0.73 +0.05 +0.23 / 3.3  8.7 7.6 cm
T1 seed 0  step 01     10/50            15/50            val end 0.304
           step 02     5/50 (4090 5)    10/50 (4090 14)  val end 0.531 (4090 0.544)
```

**What it buys:** T1-wide n_exec 8 goes 18 → 29 / 150. Seed 0 finds the cube on both GPUs (x slope +0.73 / +0.74, miss ~3 cm), where at a constant rate it was blind.

**Where it fails:** it doesn't make runs find the cube more often: 1 of 3 sees on both steps, and which seed it is changes (step 01: seed 2; step 02: seed 0). n_exec 16 drops 21 → 14. On T1 (10 demos) it overfits harder: val is lowest at 2k steps (0.17) and climbs to 0.53, n_exec 8 halves (10 → 5).

**Why most episodes fail** (`experiments/verne/scripts/reach_track.py`, seeds 0 and 1, 10 episodes each): in every failure the gripper drives forward (away from the robot) to x ≈ +16-17 cm, 6 cm past the furthest cube spawn, and parks there 11-24 cm beyond the cube; seed 0 never comes back (≤ 1.7 cm). Successes stop near x +1.5 cm and grasp. A blind run scores like a seeing one because its forward spot sometimes sits on the cube. No LR schedule addresses that.

**Later:** cosine with warmup, and EMA weights (the rest of box 11).